In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/songs.csv')
df

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...
...,...,...,...,...
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...


In [3]:
df.isnull().sum()

,0
artist,0
song,0
link,0
text,0


In [4]:
df['text'][0]

"Look at her face, it's a wonderful face  \r\nAnd it means something special to me  \r\nLook at the way that she smiles when she sees me  \r\nHow lucky can one fellow be?  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?  \r\n  \r\nAnd when we go for a walk in the park  \r\nAnd she holds me and squeezes my hand  \r\nWe'll go on walking for hours and talking  \r\nAbout all the things that we plan  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?\r\n\r\n"

In [5]:
df['text'] = (df['text'].str.lower().replace(r'^\w\s', ' ', regex=True).replace(r'\n', ' ', regex=True))


In [6]:
import nltk
nltk.download('punkt_tab')  # or 'punkt_tab' in newer versions

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [ ]:
#df['text'] = df['text'].apply(lambda x: tokenization(x))

KeyboardInterrupt: 

In [8]:
!pip install swifter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=d816e3ebdfd042766c2b1a2715155e976782fa1fc380d336aee83785ab6bb151
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter


In [9]:
import swifter
df['text'] = df['text'].swifter.apply(tokenization)

Pandas Apply:   0%|          | 0/57650 [00:00<?, ?it/s]

In [ ]:
"""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)
"""

"\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\ntfidvector = TfidfVectorizer(analyzer='word',stop_words='english')\nmatrix = tfidvector.fit_transform(df['text'])\nsimilarity = cosine_similarity(matrix)\n"

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

tfidvector = TfidfVectorizer(analyzer='word',
                             stop_words='english',
                             max_features=5000)

matrix = tfidvector.fit_transform(df['text'])


nn = NearestNeighbors(metric='cosine', algorithm='brute')
nn.fit(matrix)

# find top 6 neighbors (1 = itself + 5 recommendations)
distances, indices = nn.kneighbors(matrix, n_neighbors=6)


In [ ]:
def recommend(song_index, n_recs=5):
    # find n_recs+1 neighbors (including the song itself)
    distances, indices = nn.kneighbors(matrix[song_index], n_neighbors=n_recs+1)

    print(f"Song: {df.iloc[song_index]['song']}")
    print("\nRecommended songs:")

    # skip the first index (because it's the song itself)
    for i in range(1, len(indices[0])):
        idx = indices[0][i]
        print(f"- {df.iloc[idx]['song']}  (distance: {distances[0][i]:.3f})")


In [ ]:
recommend(42, n_recs=5)

Song: If It Wasn't For The Nights

Recommended songs:
- Last Night  (distance: 0.315)
- Through The Night  (distance: 0.422)
- One Night Stand  (distance: 0.465)
- The Night Inside Me  (distance: 0.492)
- All Through The Night  (distance: 0.501)


In [11]:
def recommend2(song_title, n_recs=5):
    # find the index of the song in your dataframe
    try:
        song_index = df[df['song'].str.lower() == song_title.lower()].index[0]
    except IndexError:
        print("❌ Song not found in the dataset.")
        return

    # get nearest neighbors
    distances, indices = nn.kneighbors(matrix[song_index], n_neighbors=n_recs+1)

    print(f"Song: {df.iloc[song_index]['song']}")
    print("\nRecommended songs:")

    for i in range(1, len(indices[0])):  # skip the first (itself)
        idx = indices[0][i]
        print(f"- {df.iloc[idx]['song']}  (distance: {distances[0][i]:.3f})")


In [12]:
recommend2("love", n_recs=5)

Song: Love

Recommended songs:
- Loved  (distance: 0.555)
- Love Them Girls  (distance: 0.570)
- Still In Love  (distance: 0.570)
- It's So Cool  (distance: 0.575)
- You  (distance: 0.579)


In [13]:
!pip install streamlit pyngrok spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.1/356.1 kB 34.2 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.0
    Uninstalling cachetools-7.0.0:
      Successfully uninstalled cachetools-7.0.0


In [14]:
df.to_csv("songs_processed.csv", index=False)

In [15]:
%%writefile app_song.py
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# --------------------------
# 1. Load dataset
# --------------------------
df = pd.read_csv("songs_processed.csv")

# --------------------------
# 2. TF-IDF + Nearest Neighbors
# --------------------------
tfidf = TfidfVectorizer(analyzer='word', stop_words='english')
matrix = tfidf.fit_transform(df['song'])  # lyrics column
nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(matrix)

def recommend(song_title, n_recs=5):
    try:
        song_index = df[df['song'].str.lower() == song_title.lower()].index[0]
    except IndexError:
        return []  # Song not found

    distances, indices = nn.kneighbors(matrix[song_index], n_neighbors=n_recs+1)
    recs = []
    for i in range(1, len(indices[0])):  # skip the song itself
        idx = indices[0][i]
        recs.append({
            "title": df.iloc[idx]['song'],
            "artist": df.iloc[idx]['artist'] if 'artist' in df.columns else None
        })
    return recs

# --------------------------
# 3. Spotify API setup
# --------------------------
client_id = "4645457766ca483fb6616b898464ac30"
client_secret = "80cc636368a64a95943803beef505847"

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
))

def get_album_cover(song_title, artist=None):
    query = song_title if not artist else f"{song_title} {artist}"
    results = sp.search(q=query, limit=1, type='track')
    if results['tracks']['items']:
        return results['tracks']['items'][0]['album']['images'][0]['url']
    return None

# --------------------------
# 4. Streamlit frontend
# --------------------------
st.title("🎶 Song Recommendation System")

song_name = st.text_input("Enter a song title:")

if st.button("Recommend"):
    if song_name:
        recs = recommend(song_name, n_recs=5)
        if not recs:
            st.error("❌ Song not found in dataset.")
        else:
            st.subheader(f"Recommendations for: {song_name}")
            for rec in recs:
                col1, col2 = st.columns([1,3])
                with col1:
                    cover_url = get_album_cover(rec['title'], rec.get('artist'))
                    if cover_url:
                        st.image(cover_url, width=150)
                    else:
                        st.write("No cover found")
                with col2:
                    st.write(f"**{rec['title']}**")
                    if rec.get('artist'):
                        st.write(f"Artist: {rec['artist']}")


Writing app_song.py


In [16]:
from pyngrok import ngrok
ngrok.set_auth_token("39krqDLb2aGiZCnEj5cMvvImPAd_7jhAU2BntG1wvWaiKkUBU")

In [22]:
!streamlit run app_song.py &>/dev/null&

from pyngrok import ngrok
public_url = ngrok.connect(8501, "http")
print("✅ Streamlit app running on:", public_url)

✅ Streamlit app running on: NgrokTunnel: "https://8539-136-109-178-208.ngrok-free.app" -> "http://localhost:8501"


In [24]:
from google.colab import files
files.download('app_song.py')
files.download('songs_processed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>